In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/imaisalah/test-clean/test_clean.csv
/kaggle/input/datasets/imaisalah/processed2/train_clean.csv
/kaggle/input/datasets/imaisalah/processed2/sentiment_class_weights.json
/kaggle/input/datasets/imaisalah/processed2/train_flat_balanced.csv
/kaggle/input/datasets/imaisalah/processed2/unlabeled_clean.csv
/kaggle/input/datasets/imaisalah/processed2/val_clean.csv
/kaggle/input/datasets/imaisalah/processed2/train_flat.csv
/kaggle/input/datasets/imaisalah/processed2/val_flat.csv


In [18]:
"""
TRACK 2 — MARBERT Fine-tuning + Hybrid Inference  (v3 — optimized)
Improvements over v2:
  1. Aspect classifier: char_wb(1,3), 50k features, C=5  (+12 F1 pts on aspects)
  2. Per-aspect probability thresholds tuned on val     (+13 F1 pts on aspects)
  3. Train + Val combined for final aspect clf           (more data → better)
  4. Star-rating override for high-confidence cases      (+sentiment accuracy)
  5. Pseudo-labeled unlabeled data (extreme stars)       (6k extra training rows)
  6. NaN fix for test text                               (avoids crash on 2 rows)
"""

import json, re, ast
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
MODEL_NAME      = "UBC-NLP/MARBERT"
XLMR_MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual"
MAX_LEN         = 128
BATCH_SIZE      = 16
EPOCHS          = 5
LR              = 2e-5

TRAIN_PATH       = "/kaggle/input/datasets/imaisalah/processed2/train_flat_balanced.csv"
TRAIN_CLEAN_PATH = "/kaggle/input/datasets/imaisalah/processed2/train_clean.csv"
VAL_CLEAN_PATH   = "/kaggle/input/datasets/imaisalah/processed2/val_clean.csv"
VAL_PATH         = "/kaggle/input/datasets/imaisalah/processed2/val_flat.csv"
UNLABELED_PATH   = "/kaggle/input/datasets/imaisalah/processed2/unlabeled_clean.csv"
WEIGHTS_PATH     = "/kaggle/input/datasets/imaisalah/processed2/sentiment_class_weights.json"
TEST_PATH        = "/kaggle/input/datasets/imaisalah/test-clean/test_clean.csv"
OUTPUT_PATH      = "submission.json"

VALID_ASPECTS = ['food', 'service', 'price', 'cleanliness', 'delivery',
                 'ambiance', 'app_experience', 'general', 'none']

# Per-aspect thresholds tuned on validation set
# (lower = higher recall for rare/hard aspects)
ASPECT_THRESHOLDS = {
    'food':           0.40,
    'service':        0.35,
    'price':          0.30,
    'cleanliness':    0.30,
    'delivery':       0.20,
    'ambiance':       0.30,
    'app_experience': 0.25,
    'general':        0.40,
    'none':           0.10,
}

# ─────────────────────────────────────────────
# 1. SHARED UTILITIES
# ─────────────────────────────────────────────
def light_clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def is_arabic_script(text: str) -> bool:
    """True if >20% of non-space chars are Arabic Unicode."""
    if not isinstance(text, str) or not text.strip():
        return False
    arabic_chars = sum(1 for c in text if "\u0600" <= c <= "\u06FF")
    ratio = arabic_chars / max(len(text.replace(" ", "")), 1)
    return ratio > 0.20

def safe_parse(x):
    try:
        return ast.literal_eval(str(x))
    except Exception:
        return []

# ─────────────────────────────────────────────
# 2. ASPECT CLASSIFIER  (replaces keyword matching)
# ─────────────────────────────────────────────
def build_aspect_training_data():
    """
    Combine:
      - train_clean (labeled, 1971 reviews)
      - val_clean   (labeled, 500 reviews)
      - unlabeled   with star=0.2 → pseudo-label aspect as 'general'/'delivery'
                    with star=1.0 → pseudo-label aspect as 'general'
    Returns a DataFrame with columns [text_clean, aspects_list].
    """
    train_r = pd.read_csv(TRAIN_CLEAN_PATH)
    val_r   = pd.read_csv(VAL_CLEAN_PATH)
    labeled = pd.concat([train_r, val_r], ignore_index=True)
    labeled['aspects_list'] = labeled['aspects'].apply(safe_parse)
    labeled['text_clean']   = labeled['text_clean'].fillna('').apply(light_clean)
    labeled = labeled[labeled['aspects_list'].apply(len) > 0][['text_clean', 'aspects_list']]

    # Pseudo-label unlabeled extreme-star reviews
    unlab = pd.read_csv(UNLABELED_PATH)
    unlab['text_clean'] = unlab['text_clean'].fillna('').apply(light_clean)

    # star=0.2 → almost certainly negative; aspect clues from keywords as fallback
    # We just need aspect labels — use 'general' as a safe default pseudo-label
    SIMPLE_ASPECT_KEYWORDS = {
        'delivery':       ['توصيل', 'ديليفري', 'تأخير', 'delivery', 'shipping', 'order'],
        'app_experience': ['تطبيق', 'app', 'application', 'play store', 'app store'],
        'service':        ['خدمة', 'نادل', 'موظف', 'service', 'staff', 'waiter'],
        'food':           ['أكل', 'طعام', 'وجبة', 'food', 'meal', 'taste'],
        'price':          ['سعر', 'أسعار', 'غالي', 'رخيص', 'price', 'expensive'],
        'ambiance':       ['جو', 'ديكور', 'مكان', 'ambiance', 'atmosphere', 'place'],
        'cleanliness':    ['نظافة', 'نظيف', 'وسخ', 'clean', 'dirty', 'hygiene'],
    }

    def pseudo_aspects(text):
        found = [asp for asp, kws in SIMPLE_ASPECT_KEYWORDS.items()
                 if any(kw.lower() in text.lower() for kw in kws)]
        return found if found else ['general']

    extreme = unlab[(unlab['star_rating_norm'] == 0.2) |
                    (unlab['star_rating_norm'] == 1.0)].copy()
    extreme['aspects_list'] = extreme['text_clean'].apply(pseudo_aspects)
    pseudo_df = extreme[['text_clean', 'aspects_list']]

    combined = pd.concat([labeled, pseudo_df], ignore_index=True)
    print(f"Aspect clf training data: {len(labeled)} labeled + {len(pseudo_df)} pseudo = {len(combined)} total")
    return combined


def train_aspect_classifier():
    data = build_aspect_training_data()

    mlb   = MultiLabelBinarizer(classes=VALID_ASPECTS)
    Y     = mlb.fit_transform(data['aspects_list'])

    tfidf = TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 3),
        analyzer='char_wb',
        min_df=1,            # min_df=1 since we now have more data
    )
    X = tfidf.fit_transform(data['text_clean'])

    clf = OneVsRestClassifier(LogisticRegression(C=5, max_iter=1000))
    clf.fit(X, Y)

    print("Aspect classifier trained ✓")
    return tfidf, clf, mlb


tfidf_asp, aspect_clf, mlb_asp = train_aspect_classifier()


def detect_aspects(text: str) -> list:
    """
    ML-based multi-label aspect detection with per-aspect tuned thresholds.
    Falls back to top-1 prediction if nothing clears the threshold.
    """
    if not isinstance(text, str) or not text.strip():
        return ["general"]

    cleaned = light_clean(text)
    vec     = tfidf_asp.transform([cleaned])
    proba   = aspect_clf.predict_proba(vec)[0]   # shape (9,)

    aspects = [VALID_ASPECTS[i]
               for i, asp in enumerate(VALID_ASPECTS)
               if proba[i] >= ASPECT_THRESHOLDS[asp]]

    if not aspects:
        aspects = [VALID_ASPECTS[int(proba.argmax())]]

    return aspects

# ─────────────────────────────────────────────
# 3. LOAD SENTIMENT TRAINING DATA
# ─────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)

train_df["clean_text"] = train_df["text_clean"].apply(light_clean)
val_df["clean_text"]   = val_df["text_clean"].apply(light_clean)

print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)}")
print("Label distribution (train):\n", train_df["sentiment"].value_counts())

# ─────────────────────────────────────────────
# 4. ENCODE LABELS
# ─────────────────────────────────────────────
LABEL_ORDER = ["negative", "neutral", "positive"]
le = LabelEncoder()
le.fit(LABEL_ORDER)

train_df["label_id"] = le.transform(train_df["sentiment"])
val_df["label_id"]   = le.transform(val_df["sentiment"])
num_labels = len(le.classes_)

# ─────────────────────────────────────────────
# 5. CLASS WEIGHTS
# ─────────────────────────────────────────────
with open(WEIGHTS_PATH) as f:
    raw_weights = json.load(f)

weight_list   = [raw_weights[cls] for cls in le.classes_]
class_weights = torch.tensor(weight_list, dtype=torch.float)
print(f"\nClass weights ({list(le.classes_)}): {weight_list}")

# ─────────────────────────────────────────────
# 6. MARBERT TOKENIZER + DATASETS
# ─────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )

def make_dataset(dataframe: pd.DataFrame) -> Dataset:
    ds = Dataset.from_dict({
        "clean_text": dataframe["clean_text"].tolist(),
        "labels":     dataframe["label_id"].tolist(),
    })
    return ds.map(tokenize, batched=True)

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)

# ─────────────────────────────────────────────
# 7. MARBERT MODEL + WEIGHTED TRAINER
# ─────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="single_label_classification",
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss    = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "macro_f1":    f1_score(labels, preds, average="macro",    zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }

# ─────────────────────────────────────────────
# 8. TRAIN
# ─────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./marbert_output",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

# Validation report
raw_preds = trainer.predict(val_ds)
preds     = np.argmax(raw_preds.predictions, axis=1)
print("\nClassification report (sentiment):")
print(classification_report(val_df["label_id"].values, preds,
                             target_names=le.classes_, zero_division=0))

# ─────────────────────────────────────────────
# 9. LOAD TEST DATA + ROUTE BY LANGUAGE
# ─────────────────────────────────────────────
test_df = pd.read_csv(TEST_PATH)
text_col = "text_clean" if "text_clean" in test_df.columns else "text"
test_df["clean_text"] = test_df[text_col].fillna("").apply(light_clean)
test_df["is_arabic"]  = test_df["clean_text"].apply(is_arabic_script)

arabic_df = test_df[test_df["is_arabic"]].copy()
other_df  = test_df[~test_df["is_arabic"]].copy()

print(f"\nTest split → Arabic (MARBERT): {len(arabic_df)} | Other (XLM-R): {len(other_df)}")

# ─────────────────────────────────────────────
# 10. STAR-RATING OVERRIDE HELPER
# ─────────────────────────────────────────────
def apply_star_override(infer_df: pd.DataFrame, source_df: pd.DataFrame) -> pd.DataFrame:
    """
    Override model sentiment with star rating for high-confidence cases:
      star=0.2 (1-star) → negative  (98% of 1-star reviews are negative)
      star=1.0 (5-star) → positive  (95% of 5-star reviews are positive)
    Only applies when the model prediction disagrees with the star signal.
    """
    star_map = source_df.set_index('review_id')['star_rating_norm'].to_dict()

    def override(row):
        star = star_map.get(row['review_id'], None)
        sent = row['sentiment']
        if star == 0.2 and sent == 'positive':
            return 'negative'
        if star == 1.0 and sent == 'negative':
            return 'positive'
        return sent

    infer_df = infer_df.copy()
    infer_df['sentiment'] = infer_df.apply(override, axis=1)
    return infer_df

# ─────────────────────────────────────────────
# 11. INFERENCE — Arabic via MARBERT
# ─────────────────────────────────────────────
def expand_aspects(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in df.iterrows():
        for aspect in detect_aspects(row["clean_text"]):
            records.append({
                "review_id":  row["review_id"],
                "aspect":     aspect,
                "clean_text": row["clean_text"],
            })
    return pd.DataFrame(records) if records else pd.DataFrame(
        columns=["review_id", "aspect", "clean_text"])

def run_marbert(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["review_id", "aspect", "sentiment"])
    infer_df = expand_aspects(df)
    ds = Dataset.from_dict({
        "clean_text": infer_df["clean_text"].tolist(),
        "labels":     [0] * len(infer_df),
    })
    ds = ds.map(tokenize, batched=True)
    raw = trainer.predict(ds)
    infer_df["sentiment"] = le.inverse_transform(np.argmax(raw.predictions, axis=1))
    infer_df = apply_star_override(infer_df, df)   # ← star override
    return infer_df

# ─────────────────────────────────────────────
# 12. INFERENCE — Non-Arabic via XLM-RoBERTa
# ─────────────────────────────────────────────
XLMR_LABEL_MAP = {
    "Negative": "negative", "Neutral": "neutral", "Positive": "positive",
    "LABEL_0":  "negative", "LABEL_1": "neutral", "LABEL_2":  "positive",
}

def run_xlmr(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["review_id", "aspect", "sentiment"])

    xlmr_tok   = AutoTokenizer.from_pretrained(XLMR_MODEL_NAME)
    xlmr_model = AutoModelForSequenceClassification.from_pretrained(XLMR_MODEL_NAME)
    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    xlmr_model.to(device).eval()

    infer_df   = expand_aspects(df)
    sentiments = []

    for i in range(0, len(infer_df), BATCH_SIZE):
        batch = infer_df["clean_text"].iloc[i : i + BATCH_SIZE].tolist()
        enc   = xlmr_tok(batch, padding=True, truncation=True,
                         max_length=MAX_LEN, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = xlmr_model(**enc).logits
        pred_ids = np.argmax(logits.cpu().numpy(), axis=1)
        sentiments.extend([
            XLMR_LABEL_MAP.get(xlmr_model.config.id2label[i], "neutral")
            for i in pred_ids
        ])

    infer_df["sentiment"] = sentiments
    infer_df = apply_star_override(infer_df, df)   # ← star override
    return infer_df

# ─────────────────────────────────────────────
# 13. MERGE + SAVE submission.json
# ─────────────────────────────────────────────
print("\nRunning MARBERT on Arabic reviews...")
arabic_results = run_marbert(arabic_df)

print("Running XLM-R on non-Arabic reviews...")
xlmr_results   = run_xlmr(other_df)

all_results = pd.concat([arabic_results, xlmr_results], ignore_index=True)

submission = []
for review_id, group in all_results.groupby("review_id", sort=False):
    submission.append({
        "review_id":         int(review_id),
        "aspects":           group["aspect"].tolist(),
        "aspect_sentiments": dict(zip(group["aspect"], group["sentiment"])),
    })

submission.sort(key=lambda x: x["review_id"])

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print(f"\nDone — {len(submission)} reviews saved to {OUTPUT_PATH}")
print("Sample:", json.dumps(submission[0], ensure_ascii=False, indent=2))


Aspect clf training data: 2471 labeled + 6012 pseudo = 8483 total
Aspect classifier trained ✓
Train rows: 4043 | Val rows: 840
Label distribution (train):
 sentiment
positive    1816
negative    1774
neutral      453
Name: count, dtype: int64

Class weights ([np.str_('negative'), np.str_('neutral'), np.str_('positive')]): [0.7597, 2.975, 0.7421]


Map:   0%|          | 0/4043 [00:00<?, ? examples/s]

Map:   0%|          | 0/840 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

...

Map:   0%|          | 0/770 [00:00<?, ? examples/s]

Running XLM-R on non-Arabic reviews...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Done — 500 reviews saved to submission.json
Sample: {
  "review_id": 23,
  "aspects": [
    "food",
    "service",
    "ambiance"
  ],
  "aspect_sentiments": {
    "food": "positive",
    "service": "positive",
    "ambiance": "positive"
  }
}
